In [2]:
import os
import psycopg2
import numpy as np
from langchain.embeddings import OllamaEmbeddings
from langchain.vectorstores.pgvector import PGVector

In [3]:
texts = [
    "The Constitution of the United States is the supreme law of the land.",
    "It was adopted on September 17, 1787, and ratified by the states in 1788.",
    "The Constitution establishes the framework for the federal government and its relationship with the states.",
    "It consists of a preamble, seven articles, and 27 amendments.",
    "The Bill of Rights, which includes the first ten amendments, guarantees individual freedoms and rights."
]

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
embedding_list = []

for text in texts:
    embedding_list.append(embeddings.embed_query(text))

C:\Users\sneha\AppData\Local\Temp\ipykernel_6380\685475438.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [5]:
len(embedding_list), len(embedding_list[0])

(5, 768)

In [6]:
conn = psycopg2.connect("dbname=vectordb user=postgres password=5107")
cur = conn.cursor()

for i in range(len(embedding_list)):
    embedding = embedding_list[i]
    content = texts[i]
    cur.execute(
        "INSERT INTO items (content, embedding) VALUES (%s, %s)",
        (content, embedding)
    )

conn.commit()
cur.close()
conn.close()

In [7]:
new_text = "The Constitution is a living document that can be amended to address changing societal needs."
new_embedding = embeddings.embed_query(new_text)

In [8]:
conn = psycopg2.connect("dbname=vectordb user=postgres password=5107")
cur = conn.cursor()
cur.execute("""
    SELECT content, embedding
    FROM items
    ORDER BY embedding <-> %s::vector
    LIMIT 5
""", (new_embedding,))

In [9]:
results = cur.fetchall()
for result in results:
    content, embedding = result
    print(f"Content: {content}\nEmbedding: {embedding}\n")

Content: The Constitution establishes the framework for the federal government and its relationship with the states.
Embedding: [1.2433711,1.9568248,-3.655692,-0.9125598,1.6082318,0.40682048,-0.6472439,0.63135386,1.7378945,-1.4919983,-0.58262545,-0.48703122,1.406442,0.5054609,0.9102643,0.07375108,0.2746587,-0.408066,0.28327957,0.8028949,-1.1860875,1.0835981,-0.1990326,0.40824488,1.4164398,1.3046476,-0.11160454,-0.855588,-0.6647558,-0.3496515,-0.5628142,0.7626246,0.5950224,-0.3815587,0.08825204,-2.749468,0.3935452,0.8501365,0.6941671,-1.4752972,-0.3032159,0.20814712,-1.842574,-1.2128488,0.950872,-0.63942164,1.2221515,0.21416111,1.0635679,0.14455247,0.05366453,-0.7590128,0.5989508,-0.59886396,0.015785074,0.4329471,0.42607477,0.8189571,-0.42682347,0.09188611,2.1224127,0.8013808,-0.9970031,1.3420128,0.03533275,0.5849103,-0.36989602,2.1026316,0.34288043,-0.1726889,0.3008276,-0.17178877,0.24615915,0.24281721,-0.33967453,-0.054206602,-0.4547225,-0.2505946,-0.6596337,-0.3295116,1.9185224,-1.01